# Redrob Candidate Ranking System
## Team Clover - Colab Sandbox Demo
This notebook deploys our ACTUAL project code into the Colab environment.
**Prerequisite:** Ensure you have uploaded `test_candidates.jsonl` to your Colab workspace.

In [ ]:
!pip install -q transformers torch sentence-transformers pandas

In [ ]:
%%writefile download_model.py
import os
from sentence_transformers import SentenceTransformer, CrossEncoder

def download_model():
    print("Downloading all-MiniLM-L6-v2 model for offline use...")
    model_name = "all-MiniLM-L6-v2"
    local_dir = os.path.join(os.path.dirname(__file__), "local_model", model_name)
    
    # Download and cache the model locally
    model = SentenceTransformer(model_name)
    model.save(local_dir)
    print(f"Bi-Encoder successfully saved to {local_dir}")
    
    print("Downloading ms-marco-MiniLM-L-6-v2 Cross-Encoder for offline use...")
    cross_model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    local_cross_dir = os.path.join(os.path.dirname(__file__), "local_model", "ms-marco-MiniLM-L-6-v2")
    
    cross_encoder = CrossEncoder(cross_model_name)
    cross_encoder.save(local_cross_dir)
    print(f"Cross-Encoder successfully saved to {local_cross_dir}")

    print("You can now safely run main.py completely offline.")

if __name__ == "__main__":
    download_model()


In [ ]:
%%writefile data_loader.py
import json
import logging

logger = logging.getLogger(__name__)

def load_candidates_stream(filepath):
    """
    Generator that stream-loads candidates from the JSONL file.
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        candidate = json.loads(line)
                        yield candidate
                    except json.JSONDecodeError as e:
                        logger.error('Failed to decode line in candidates.jsonl', extra={"error": str(e)})
                        continue
    except FileNotFoundError:
        logger.error('candidates.jsonl not found', extra={"filepath": filepath})
        raise

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    count = 0
    import sys
    if len(sys.argv) > 1:
        filepath = sys.argv[1]
    else:
        filepath = 'candidates.jsonl'
        
    for _ in load_candidates_stream(filepath):
        count += 1
    print(f"Loaded {count} candidates.")


In [ ]:
%%writefile plausibility_filter.py
import datetime

CURRENT_YEAR = 2026


def is_honeypot(candidate) -> tuple[bool, str | None]:
    """
    Returns (True, reason_code) ONLY for mathematically impossible profiles.
    ~80 real honeypots in dataset. Everything else is a scoring penalty.
    """
    profile = candidate.get("profile", {})
    yoe = profile.get("years_of_experience", 0)
    skills = candidate.get("skills", [])
    career_history = candidate.get("career_history", [])

    # Check 1: Expert proficiency in 5+ skills with 0 months duration
    # Spec literally names this pattern.
    expert_zero = sum(
        1 for s in skills
        if s.get("proficiency") == "expert" and s.get("duration_months", 0) == 0
    )
    if expert_zero >= 5:
        return True, "check_1_expert_zero_months"

    # Check 2: Claimed YoE wildly exceeds career history
    # Only kill if difference is extreme (>8 years unaccounted)
    total_career_months = sum(job.get("duration_months", 0) for job in career_history)
    if total_career_months > 0 and (yoe * 12) > (total_career_months + 96):
        return True, "check_2_yoe_exceeds_career"

    # Check 3: Overlapping dates — claiming 1.5x+ more duration than calendar time
    if career_history:
        try:
            min_date = None
            max_date = None
            for job in career_history:
                start_str = job.get("start_date")
                if start_str:
                    s_date = datetime.datetime.strptime(start_str, "%Y-%m-%d")
                    if min_date is None or s_date < min_date:
                        min_date = s_date

                end_str = job.get("end_date")
                is_current = job.get("is_current", False)
                if is_current or not end_str:
                    e_date = datetime.datetime.now()
                else:
                    e_date = datetime.datetime.strptime(end_str, "%Y-%m-%d")
                if max_date is None or e_date > max_date:
                    max_date = e_date

            if min_date and max_date:
                chronological_months = (max_date.year - min_date.year) * 12 + (max_date.month - min_date.month)
                if chronological_months > 0 and total_career_months > (chronological_months * 2.0 + 36):
                    return True, "check_3_overlapping_career_dates"
        except ValueError:
            pass

    # Check 4: Time-traveling tech — claiming impossible duration for recent tech
    for s in skills:
        name = s.get("name", "").lower()
        dur = s.get("duration_months", 0)
        if name == "qlora" and dur > 38:
            return True, "check_4_time_traveling_tech"
        if name == "langchain" and dur > 50:
            return True, "check_4_time_traveling_tech"
        if name == "chatgpt" and dur > 46:
            return True, "check_4_time_traveling_tech"
        if name == "llama-2" and dur > 38:
            return True, "check_4_time_traveling_tech"

    return False, None


def get_penalty_reasons(candidate) -> list[str]:
    """
    Returns list of penalty reason codes for heuristic red flags.
    These are NOT honeypots — scorer.py applies multipliers.
    """
    reasons = []
    profile = candidate.get("profile", {})
    skills = candidate.get("skills", [])
    career_history = candidate.get("career_history", [])
    yoe = profile.get("years_of_experience", 0)

    career_text = " ".join([job.get("description", "").lower() for job in career_history])
    summary_text = profile.get("summary", "").lower()
    full_text = career_text + " " + summary_text

    # Duplicate descriptions across roles
    if len(career_history) >= 2:
        descriptions = [job.get("description", "").strip().lower() for job in career_history]
        valid = [d for d in descriptions if len(d) > 40]
        if len(set(valid)) < len(valid):
            reasons.append("duplicate_descriptions")

    # Skill duration exceeds career by large margin
    total_career_months = sum(job.get("duration_months", 0) for job in career_history)
    max_skill_dur = max((s.get("duration_months", 0) for s in skills), default=0)
    if max_skill_dur > (total_career_months + 48):
        reasons.append("skill_duration_exceeds_career")

    # CV/speech without NLP/IR
    title = profile.get("current_title", "").lower()
    if any(t in title for t in ["computer vision", "cv engineer", "robotics", "speech engineer"]):
        nlp_signals = ["nlp", "retrieval", "ranking", "embedding", "transformer", "vector", "natural language", "llm", "search"]
        if not any(sig in full_text for sig in nlp_signals):
            reasons.append("cv_without_nlp")

    # Fake proficiency: expert claim but terrible assessment
    skill_assessments = candidate.get("redrob_signals", {}).get("skill_assessment_scores", {})
    for s in skills:
        name = s.get("name")
        prof = s.get("proficiency", "beginner")
        if name in skill_assessments:
            sc = skill_assessments[name]
            if prof == "expert" and sc < 40:
                reasons.append("fake_proficiency")
                break
            if prof == "advanced" and sc < 25:
                reasons.append("fake_proficiency")
                break

    # Wrong persona: "enthusiast"/"aspiring" with 3+ years
    headline = profile.get("headline", "").lower()
    if ("enthusiast" in headline or "aspiring" in headline) and yoe > 3:
        reasons.append("wrong_persona_enthusiast")

    # Transitioning persona
    if "interested in transitioning toward" in full_text and "learning modern ml practice" in full_text:
        reasons.append("transitioning_persona")

    # LangChain-only without foundations
    skill_names_lower = {s.get("name", "").lower() for s in skills}
    llm_wrappers = {"langchain", "llamaindex", "openai", "chatgpt"}
    core_ml = {
        "embedding", "embeddings", "faiss", "vector", "retrieval",
        "ranking", "sentence-transformers", "nlp", "information retrieval",
        "recommendation", "search", "bert", "pytorch", "tensorflow"
    }
    if (any(s in skill_names_lower for s in llm_wrappers)
        and not any(s in skill_names_lower for s in core_ml)
        and not any(kw in career_text for kw in [
            "retrieval", "ranking", "embedding", "search engine",
            "recommendation", "nlp", "neural", "production ml"
        ])):
        reasons.append("langchain_only")

    # CV disqualifier (explicit admission)
    if "professional experience there is limited" in full_text and "transitioning toward nlp" in full_text:
        reasons.append("cv_disqualifier_explicit")
    if "most of my project work has been in cv" in full_text:
        reasons.append("cv_disqualifier_explicit")

    return reasons


In [ ]:
%%writefile scorer.py
import datetime
import re
import string

# is_valid_match has been moved down below JD_SKILLS_NORM definitions

SKILL_SYNONYMS = {
    "vector db": "vector database",
    "elastic": "elasticsearch",
    "knn": "vector search",
    "k nearest neighbor": "vector search",
    "k nearest neighbors": "vector search",
    "hf": "hugging face",
    "huggingface": "hugging face",
    "llms": "llm",
    "sklearn": "scikit learn",
    "scikitlearn": "scikit learn",
    "chatgpt": "gpt",
    "k8s": "kubernetes"
}

SYN_PATTERN = re.compile(r'\b(' + '|'.join(map(re.escape, SKILL_SYNONYMS.keys())) + r')\b')


def normalize_text(text: str) -> str:
    text = text.lower()
    text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
    text = ' '.join(text.split())
    return SYN_PATTERN.sub(lambda m: SKILL_SYNONYMS[m.group(1)], text)


TARGET_CITIES = {
    "pune", "noida", "delhi", "new delhi", "ncr", "mumbai", "hyderabad",
}

# ============================================================================
# Title relevance - hard gate against irrelevant job titles
# ============================================================================
STRONG_TITLE_MATCHES = {
    "machine learning", "ml engineer", "ml ", "deep learning", "nlp",
    "natural language", "ai engineer", "ai research", "search engineer",
    "ranking engineer", "recommendation", "data scientist", "applied ml",
    "research engineer", "research scientist", "applied scientist",
}

SENIOR_TITLE_KEYWORDS = {
    "senior", "staff", "lead", "principal", "head",
}

JUNIOR_TITLE_KEYWORDS = {
    "junior", "intern", "trainee", "fresher", "associate",
}

WEAK_TITLE_MATCHES = {
    "software engineer", "backend engineer", "data engineer", "platform engineer",
    "devops", "full stack", "python developer", "infrastructure",
}

IRRELEVANT_TITLES = {
    "hr ", "human resource", "recruiter", "talent acquisition",
    "accountant", "finance", "sales", "marketing", "business development",
    "civil", "mechanical", "electrical engineer", "procurement",
    "legal", "compliance", "admin", "office manager", "receptionist",
    "content writer", "graphic design", "ui/ux", "teacher", "professor",
    "computer vision", "cv engineer", "robotics", "speech"
}

# ============================================================================
# JD skill matching - explicit overlap with the job description
# ============================================================================
JD_CORE_SKILLS = {
    "sentence-transformers", "sentence transformers", "faiss", "pinecone",
    "weaviate", "qdrant", "milvus", "opensearch", "elasticsearch",
    "embeddings", "embedding", "vector search", "vector database",
    "rag", "retrieval augmented", "retrieval-augmented",
    "ranking", "ndcg", "mrr", "learning to rank", "learning-to-rank",
    "information retrieval",
    "map",
    "mean average precision",
    "a/b test",
    "a/b testing",
    "evaluation framework",
    "eval framework",
    "eval harness",
    "evaluation harness",
    "hybrid search",
    "hybrid retrieval",
    "bge",
    "openai embeddings",
}

JD_STRONG_SKILLS = {
    "pytorch", "tensorflow", "transformers", "bert", "llm", "llms",
    "hugging face", "huggingface", "lora", "qlora", "peft",
    "nlp", "natural language processing", "xgboost",
    "deep learning", "neural network", "machine learning",
    "scikit learn", "sklearn",
}

JD_NICE_SKILLS = {
    "python", "docker", "kubernetes", "k8s", "aws", "mlops", "ci/cd",
    "distributed systems", "microservices", "langchain", "llamaindex",
    "hr tech", "hr-tech", "recruiting", "marketplace"
}

JD_CORE_SKILLS_NORM = {normalize_text(s) for s in JD_CORE_SKILLS}
JD_STRONG_SKILLS_NORM = {normalize_text(s) for s in JD_STRONG_SKILLS}
JD_NICE_SKILLS_NORM = {normalize_text(s) for s in JD_NICE_SKILLS}
JD_ALL_NORM = JD_CORE_SKILLS_NORM | JD_STRONG_SKILLS_NORM

JD_REGEX_PATTERNS = {}
for jd_set in [JD_CORE_SKILLS_NORM, JD_STRONG_SKILLS_NORM, JD_NICE_SKILLS_NORM]:
    for skill in jd_set:
        JD_REGEX_PATTERNS[skill] = re.compile(r'\b' + re.escape(skill) + r'\b')


def is_valid_match(jd_skill_norm, candidate_string_norm):
    if jd_skill_norm == candidate_string_norm:
        return True
    pattern = JD_REGEX_PATTERNS.get(jd_skill_norm)
    if pattern and pattern.search(candidate_string_norm):
        return True
    if candidate_string_norm and f" {jd_skill_norm} " in f" {candidate_string_norm} ":
        return True
    return False


def calculate_title_multiplier(candidate):
    """Returns a multiplier based on how relevant the job title is to the JD."""
    profile = candidate.get("profile", {})
    title = profile.get("current_title", "").lower()

    base_mult = 0.45
    for t in STRONG_TITLE_MATCHES:
        if t in title:
            base_mult = 1.0
            break

    if base_mult < 1.0:
        for t in WEAK_TITLE_MATCHES:
            if t in title:
                base_mult = 0.70
                break

    if base_mult < 1.0:
        for t in IRRELEVANT_TITLES:
            if t in title:
                base_mult = 0.10
                break

    # JD is for "Senior AI Engineer — Founding Team": penalize junior titles
    for jt in JUNIOR_TITLE_KEYWORDS:
        if jt in title:
            base_mult *= 0.50
            break

    # Bonus for senior-level titles (founding team needs senior judgment)
    for st in SENIOR_TITLE_KEYWORDS:
        if st in title:
            base_mult = min(1.0, base_mult * 1.10)
            break

    return base_mult


SERVICES_FIRMS = {
    "tcs", "tata consultancy services", "infosys", "wipro", "cognizant",
    "accenture", "capgemini", "hcl", "tech mahindra", "ibm", "l&t infotech",
    "lti", "mindtree", "mphasis", "syntel", "genpact", "genpact ai"
}


def calculate_services_penalty(candidate):
    """
    Graduated services penalty:
    100% services → 0.4x, >80% → 0.55x, >60% → 0.75x.
    """
    career_history = candidate.get("career_history", [])
    if not career_history:
        return 1.0

    total_months = 0
    service_months = 0

    for job in career_history:
        dur = job.get("duration_months", 0)
        total_months += dur
        company = job.get("company", "").lower()
        is_service = False
        for firm in SERVICES_FIRMS:
            if re.search(r'\b' + re.escape(firm) + r'\b', company):
                is_service = True
                break
        if is_service:
            service_months += dur

    if total_months == 0:
        return 1.0

    ratio = service_months / total_months
    if ratio >= 0.99:
        return 0.4
    if ratio > 0.80:
        return 0.55
    if ratio > 0.60:
        return 0.75
    return 1.0


def calculate_skill_match_score(candidate):
    """
    Calculates explicit skill overlap with the JD requirements.
    Returns a score from 0.0 to 1.0.
    """
    skills = candidate.get("skills", [])
    skill_names = set()
    for s in skills:
        name = s.get("name", "").lower()
        skill_names.add(name)

    career_text = ""
    for job in candidate.get("career_history", []):
        career_text += " " + job.get("description", "").lower()

    norm_career_text = normalize_text(career_text)

    core_hits = 0
    strong_hits = 0
    nice_hits = 0

    for sname in skill_names:
        norm_sname = normalize_text(sname)
        for jd_skill in JD_CORE_SKILLS_NORM:
            if is_valid_match(jd_skill, norm_sname):
                core_hits += 1
                break
        for jd_skill in JD_STRONG_SKILLS_NORM:
            if is_valid_match(jd_skill, norm_sname):
                strong_hits += 1
                break
        for jd_skill in JD_NICE_SKILLS_NORM:
            if is_valid_match(jd_skill, norm_sname):
                nice_hits += 1
                break

    # Also check career descriptions for core skill evidence
    for jd_skill in JD_CORE_SKILLS_NORM:
        if is_valid_match(jd_skill, norm_career_text):
            core_hits += 1.0

    core_score = min(1.0, core_hits / 3)
    strong_score = min(1.0, strong_hits / 3)
    nice_score = min(1.0, nice_hits / 2)

    base_score = (core_score * 0.50) + (strong_score * 0.35) + (nice_score * 0.15)

    core_role_matches = 0
    for job in candidate.get("career_history", []):
        job_desc = job.get("description", "").lower()
        norm_job_desc = normalize_text(job_desc)
        if any(is_valid_match(jd_skill, norm_job_desc) for jd_skill in JD_CORE_SKILLS_NORM):
            core_role_matches += 1

    if core_role_matches >= 2:
        base_score += 0.15

    skill_assessments = candidate.get("redrob_signals", {}).get("skill_assessment_scores", {})
    assessment_bonus = 0.0
    for sname, sscore in skill_assessments.items():
        sname_lower = normalize_text(sname)
        is_relevant = False
        for jd_skill in JD_ALL_NORM:
            if is_valid_match(jd_skill, sname_lower):
                is_relevant = True
                break
        if is_relevant:
            if sscore > 85:
                assessment_bonus += 0.1
            elif sscore > 70:
                assessment_bonus += 0.05

    return min(1.0, base_score + min(0.2, assessment_bonus))


def calculate_signal_score(candidate):
    """Calculates the behavioral signal score (max 1.0)."""
    score = 0.0
    signals = candidate.get("redrob_signals", {})
    profile = candidate.get("profile", {})

    last_active_str = signals.get("last_active_date")
    resp_rate = signals.get("recruiter_response_rate", 0.0)
    interview_rate = signals.get("interview_completion_rate", 0.0)

    days_inactive = 180
    if last_active_str:
        try:
            last_active = datetime.datetime.strptime(last_active_str, "%Y-%m-%d")
            days_inactive = max(0, (datetime.datetime(2026, 6, 16) - last_active).days)
            if days_inactive <= 30 and resp_rate > 0.5:
                score += 0.33
            elif days_inactive <= 90 and resp_rate > 0.3:
                score += 0.15
        except ValueError:
            pass

    if interview_rate > 0.8:
        score += 0.17

    offer_acceptance = signals.get("offer_acceptance_rate", -1)
    if offer_acceptance > 0.7:
        score += 0.15
    elif 0 <= offer_acceptance < 0.3:
        score -= 0.20

    notice_days = signals.get("notice_period_days", 90)
    if notice_days <= 30:
        score += 0.16
    elif notice_days <= 60:
        score += 0.08
    elif notice_days <= 90:
        score += 0.03
    else:
        # aggressive penalty for 90+
        score -= 0.15

    loc = profile.get("location", "").lower()
    will_relocate = signals.get("willing_to_relocate", False)
    is_compatible = any(city in loc for city in TARGET_CITIES)

    if is_compatible or will_relocate:
        score += 0.17

    gh_score = signals.get("github_activity_score", -1)
    if gh_score > 60:
        score += 0.25
    elif gh_score > 40:
        score += 0.10

    saved_30d = signals.get("saved_by_recruiters_30d", 0)
    if saved_30d and saved_30d > 20:
        score += 0.08
    elif saved_30d and saved_30d > 5:
        score += 0.04

    apps_30d = signals.get("applications_submitted_30d", 0)
    if apps_30d and apps_30d > 3:
        score += 0.05

    avg_resp_time = signals.get("avg_response_time_hours", 999)
    if avg_resp_time and avg_resp_time < 6:
        score += 0.05
    elif avg_resp_time and avg_resp_time < 24:
        score += 0.02

    # P3: Soft availability gate for ghost candidates
    if days_inactive > 90 and resp_rate < 0.25:
        score *= 0.5

    score = min(1.0, score)

    # Prevent junior candidates from being rescued solely by strong signal/location scores
    yoe = profile.get("years_of_experience", 0)
    if yoe < 5.0:
        score *= 0.80

    return score


def calculate_experience_band_multiplier(candidate):
    """JD says 5-9 years ideal. Apply a bell-curve-style multiplier."""
    yoe = candidate.get("profile", {}).get("years_of_experience", 0)
    if 6.0 <= yoe <= 8.0:
        return 1.05  # Absolute sweet spot for Senior Founding Team
    elif 5.0 <= yoe <= 9.0:
        return 1.0
    elif 4.0 <= yoe < 5.0:
        return 0.72
    elif 9.0 < yoe <= 11.0:
        return 0.90
    elif 3.0 <= yoe < 4.0 or 11.0 < yoe <= 14.0:
        return 0.75
    elif yoe < 3.0:
        return 0.50
    else:
        return 0.60

def calculate_startup_bonus(candidate):
    """Bonus for startup, 0-to-1, or founding experience."""
    career_text = " ".join([job.get("description", "").lower() for job in candidate.get("career_history", [])])
    if any(kw in career_text for kw in ["startup", "0 to 1", "0-to-1", "founding", "built from scratch", "early stage"]):
        return 0.05
    return 0.0


def calculate_education_bonus(candidate):
    """Small bonus for tier_1 education."""
    education = candidate.get("education", [])
    for edu in education:
        tier = edu.get("tier", "unknown")
        if tier == "tier_1":
            return 0.06
        if tier == "tier_2":
            return 0.03
    return 0.0


def calculate_certification_bonus(candidate):
    """Bonus for relevant ML/AI certifications."""
    certs = candidate.get("certifications", [])
    relevant_keywords = {
        "machine learning", "deep learning", "nlp", "natural language",
        "tensorflow", "pytorch", "ai ", "artificial intelligence",
        "data science", "aws machine", "google cloud ai",
    }
    bonus = 0.0
    for cert in certs:
        name = cert.get("name", "").lower()
        if any(kw in name for kw in relevant_keywords):
            bonus += 0.02
    return min(0.06, bonus)


DEPTH_EVIDENCE_PHRASES = [
    "deployed", "shipped", "a/b test", "offline eval",
    "ndcg", "recommendation system"
]


def calculate_depth_bonus(candidate):
    """Multiplicative bonus for candidates with deep production evidence."""
    career_text = " ".join([job.get("description", "").lower() for job in candidate.get("career_history", [])])
    hits = sum(1 for phrase in DEPTH_EVIDENCE_PHRASES if phrase in career_text)
    if hits >= 5:
        return 1.10
    return 1.0


def calculate_academic_penalty(candidate):
    """Penalize pure research/academic backgrounds with no production evidence."""
    career_text = " ".join([job.get("description", "").lower() for job in candidate.get("career_history", [])])
    academic_words = ["paper", "conference", "research", "lab", "publication", "thesis"]
    production_words = ["shipped", "deployed", "production", "users", "scale", "infrastructure"]
    
    has_academic = sum(1 for w in academic_words if w in career_text) >= 2
    has_production = any(w in career_text for w in production_words)
    
    if has_academic and not has_production:
        return 0.7
    return 1.0


def calculate_infrastructure_penalty(candidate):
    """Penalize pure MLOps/Infra engineers who lack ranking/retrieval signals."""
    career_text = " ".join([job.get("description", "").lower() for job in candidate.get("career_history", [])])
    infra_words = ["mlflow", "kubeflow", "orchestration", "pipeline", "infrastructure", "kubernetes", "docker"]
    retrieval_words = ["retrieval", "ranking", "search", "embedding", "ndcg", "recsys", "recommendation", "vector"]
    
    has_infra = sum(1 for w in infra_words if w in career_text) >= 2
    has_retrieval = any(w in career_text for w in retrieval_words)
    
    if has_infra and not has_retrieval:
        return 0.7
    return 1.0


# Penalty multipliers for heuristic red flags from plausibility_filter
PENALTY_MULTIPLIERS = {
    "duplicate_descriptions": 0.80,
    "skill_duration_exceeds_career": 0.75,
    "cv_without_nlp": 0.15,
    "fake_proficiency": 0.15,
    "wrong_persona_enthusiast": 0.15,
    "transitioning_persona": 0.15,
    "langchain_only": 0.15,
    "cv_disqualifier_explicit": 0.10,
}


def score_candidate(candidate, semantic_score, penalty_reasons=None):
    """
    Final scoring: fuses semantic similarity, explicit skill match,
    title relevance, experience band, and behavioral signals.
    """
    skill_match = calculate_skill_match_score(candidate)
    signal_score = calculate_signal_score(candidate)
    title_mult = calculate_title_multiplier(candidate)

    raw_score = (0.50 * semantic_score) + (0.30 * skill_match) + (0.20 * signal_score)

    # Education, certification, and startup bonuses (additive before multipliers)
    raw_score += calculate_education_bonus(candidate)
    raw_score += calculate_certification_bonus(candidate)
    raw_score += calculate_startup_bonus(candidate)

    final_score = raw_score * title_mult

    # Experience band fit
    final_score *= calculate_experience_band_multiplier(candidate)

    # P5: Graduated services penalty
    final_score *= calculate_services_penalty(candidate)

    # Title-chaser penalty
    career_history = candidate.get("career_history", [])
    if len(career_history) >= 3:
        durations = [j.get("duration_months", 0) for j in career_history]
        avg_tenure = sum(durations) / len(durations)
        if avg_tenure < 18:
            final_score *= 0.6

    # Hands-off architect penalty
    title = candidate.get("profile", {}).get("current_title", "").lower()
    if "architect" in title or "tech lead" in title:
        career_text = " ".join([j.get("description", "").lower() for j in career_history])
        if not any(w in career_text for w in ["code", "develop", "implement", "shipped", "python", "pytorch"]):
            final_score *= 0.6

    profile = candidate.get("profile", {})
    signals = candidate.get("redrob_signals", {})
    loc = profile.get("location", "").lower()
    country = profile.get("country", "").lower()
    will_relocate = signals.get("willing_to_relocate", False)
    is_compatible = any(city in loc for city in TARGET_CITIES)

    if not is_compatible and not will_relocate:
        if country and country != "india":
            final_score *= 0.05
        else:
            final_score *= 0.65

    last_active_str = signals.get("last_active_date")
    resp_rate = signals.get("recruiter_response_rate", 0.0)

    days_inactive = 180
    if last_active_str:
        try:
            last_active = datetime.datetime.strptime(last_active_str, "%Y-%m-%d")
            days_inactive = max(0, (datetime.datetime(2026, 6, 16) - last_active).days)
        except ValueError:
            days_inactive = 180

    if days_inactive > 180 and resp_rate < 0.1:
        final_score *= 0.1
    elif days_inactive > 180 or resp_rate < 0.1:
        final_score *= 0.5

    if not signals.get("open_to_work_flag", True):
        final_score *= 0.1

    # Academic penalty
    final_score *= calculate_academic_penalty(candidate)

    # Infrastructure penalty
    final_score *= calculate_infrastructure_penalty(candidate)

    # Apply heuristic penalties from plausibility_filter
    if penalty_reasons:
        for reason in penalty_reasons:
            mult = PENALTY_MULTIPLIERS.get(reason, 0.85)
            final_score *= mult

    # Closed source penalty
    gh_score = signals.get("github_activity_score", -1)
    yoe = profile.get("years_of_experience", 0)
    career_text = " ".join([job.get("description", "").lower() for job in career_history])
    if gh_score == 0 and yoe >= 8 and "open source" not in career_text:
        final_score *= 0.85

    # Profile completeness
    completeness = signals.get("profile_completeness_score", 50)
    if completeness < 40:
        final_score *= 0.85

    # Vector DB claim without career evidence
    skills = candidate.get("skills", [])
    vector_dbs = {"qdrant", "milvus", "pinecone", "weaviate", "faiss", "semantic search"}
    claimed_vector_dbs = [s.get("name", "").lower() for s in skills if s.get("name", "").lower() in vector_dbs]
    if len(claimed_vector_dbs) >= 2:
        if not any(db in career_text for db in vector_dbs):
            final_score *= 0.6

    # P4: Apply depth bonus multiplicatively at the end
    final_score *= calculate_depth_bonus(candidate)

    return final_score, semantic_score, signal_score, None


In [ ]:
%%writefile embedding_engine.py
import os
import numpy as np


class EmbeddingEngine:
    def __init__(self, model_path=None):
        if model_path is None:
            local_path = os.path.join(os.path.dirname(__file__), "local_model", "all-MiniLM-L6-v2")
            if os.path.isdir(local_path):
                model_path = local_path
            else:
                model_path = "all-MiniLM-L6-v2"

        # Direct Transformers/Torch inference keeps the same local MiniLM model and
        # mean-pooling behavior, but avoids SentenceTransformer's slower startup path.
        import torch
        from transformers import AutoModel, AutoTokenizer

        self.torch = torch
        try:
            torch.set_num_threads(min(8, os.cpu_count() or 4))
        except RuntimeError:
            pass

        self.tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=os.path.isdir(model_path))
        self.model = AutoModel.from_pretrained(model_path, local_files_only=os.path.isdir(model_path))
        self.model.eval()

        self.facets_text = {
            "core_ml": (
                "Built and shipped embedding-based retrieval pipelines handling millions of queries. "
                "Integrated sentence-transformers, BGE, and OpenAI embeddings into production search infrastructure. "
                "Monitored NDCG and MRR in A/B tests to improve offline-to-online correlation. "
                "Scaled hybrid retrieval architectures and managed index refresh and embedding drift."
            ),
            "engineering_infra": (
                "Deployed and maintained production vector databases like Pinecone, Weaviate, Qdrant, Milvus, and FAISS. "
                "Wrote clean, production-grade Python for large-scale distributed systems. "
                "Architected scalable infrastructure for candidate-JD matching and semantic search. "
                "Managed latency, throughput, and index optimization for real-time retrieval."
            ),
            "nice_to_haves": (
                "Fine-tuned LLMs using LoRA, QLoRA, and PEFT for domain-specific tasks. "
                "Trained learning-to-rank models using XGBoost and neural rankers. "
                "Built marketplace products and talent intelligence features in the HR-tech space. "
                "Contributed to open-source AI projects and mentored junior engineers on the team."
            )
        }

        facet_texts = [
            self.facets_text["core_ml"],
            self.facets_text["engineering_infra"],
            self.facets_text["nice_to_haves"],
        ]
        facet_embeddings = self.batch_encode(facet_texts, batch_size=3)
        self.facet_vectors = {
            "core_ml": facet_embeddings[0],
            "engineering_infra": facet_embeddings[1],
            "nice_to_haves": facet_embeddings[2],
        }

    def batch_encode(self, texts, batch_size=256):
        """
        Encodes text into normalized all-MiniLM-L6-v2 sentence embeddings.
        """
        torch = self.torch
        vectors = []
        with torch.inference_mode():
            for start in range(0, len(texts), batch_size):
                batch = texts[start:start + batch_size]
                encoded = self.tokenizer(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=256,
                    return_tensors="pt",
                )
                output = self.model(**encoded).last_hidden_state
                mask = encoded["attention_mask"].unsqueeze(-1).expand(output.size()).float()
                pooled = (output * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
                pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
                vectors.append(pooled.cpu().numpy())
        return np.vstack(vectors) if vectors else np.empty((0, 384), dtype=np.float32)

    def compute_similarity(self, candidate_embeddings):
        """
        Computes weighted similarity for a batch of candidate embeddings against JD facets.
        Returns array of similarity scores.
        """
        w_core = 0.45
        w_eng = 0.45
        w_nice = 0.10

        score_core = np.dot(candidate_embeddings, self.facet_vectors["core_ml"])
        score_eng = np.dot(candidate_embeddings, self.facet_vectors["engineering_infra"])
        score_nice = np.dot(candidate_embeddings, self.facet_vectors["nice_to_haves"])

        return (w_core * score_core) + (w_eng * score_eng) + (w_nice * score_nice)


In [ ]:
%%writefile profile_builder.py
import re

# Evidence-rich sentence detection — compiled once
_EVIDENCE_RE = re.compile(
    r'[^.]*\b(\d+%|\d+ms|\d+k|\d+x|shipped|deployed|built|production|reduced|improved|increased|launched|served|scale)\b[^.]*\.',
    re.IGNORECASE
)


def build_candidate_text(candidate):
    """
    Dense text profile optimized for 600-char embedding window.
    Front-loads: title + years, first sentence from top 2 jobs, top 5 skills by duration.
    """
    profile = candidate.get("profile", {})
    title = profile.get("current_title", "")
    yoe = profile.get("years_of_experience", 0)

    parts = [f"{yoe} years {title}"]

    # First sentence of top 2 most recent jobs
    career = candidate.get("career_history", [])
    for job in career[:2]:
        desc = job.get("description", "").strip()
        if desc:
            first = desc.split(".")[0].strip()
            if len(first) > 100:
                first = first[:97] + "..."
            parts.append(first + ".")

    # Top 5 skills by duration
    skills = candidate.get("skills", [])
    if skills:
        top = sorted(skills, key=lambda x: x.get("duration_months", 0), reverse=True)[:5]
        skill_str = ", ".join(s.get("name", "") for s in top if s.get("name"))
        if skill_str:
            parts.append(f"Skills: {skill_str}")

    return " ".join(parts)


def extract_evidence_sentence(candidate):
    """
    Extract the single most evidence-rich sentence from career history.
    Called only on shortlisted candidates (2500), not all 100k.
    Returns (sentence, company) or (None, None).
    """
    for job in candidate.get("career_history", [])[:3]:
        desc = job.get("description", "").strip()
        if not desc:
            continue
        m = _EVIDENCE_RE.search(desc)
        if m:
            sent = m.group(0).strip()
            if len(sent) > 140:
                sent = sent[:137] + "..."
            return sent, job.get("company", "")
    return None, None


In [ ]:
%%writefile reasoning_generator.py
def generate_reasoning(candidate, rank, score, semantic_score, signal_score, evidence_pair=None):
    from scorer import (
        TARGET_CITIES, JD_CORE_SKILLS_NORM, JD_STRONG_SKILLS_NORM,
        JD_NICE_SKILLS_NORM, normalize_text, is_valid_match
    )

    profile = candidate.get("profile", {})
    title = profile.get("current_title", "Professional")
    cid = candidate.get("candidate_id", "Unknown")
    yoe = profile.get("years_of_experience", 0)
    signals = candidate.get("redrob_signals", {})
    career_history = candidate.get("career_history", [])

    skill_names = set(s.get("name", "") for s in candidate.get("skills", []))
    career_text = " ".join([job.get("description", "").lower() for job in career_history])
    norm_career_text = normalize_text(career_text)

    def find_hits(jd_set, require_in_career=False):
        found = []
        for sname in skill_names:
            norm_sname = normalize_text(sname)
            for jd_skill in jd_set:
                if is_valid_match(jd_skill, norm_sname) and jd_skill not in found:
                    if require_in_career and not is_valid_match(jd_skill, norm_career_text):
                        continue
                    found.append(jd_skill)
                    break
        return found

    core_found_career = find_hits(JD_CORE_SKILLS_NORM, require_in_career=True)
    core_found_claimed = find_hits(JD_CORE_SKILLS_NORM, require_in_career=False)
    strong_found_career = find_hits(JD_STRONG_SKILLS_NORM, require_in_career=True)
    strong_found_claimed = find_hits(JD_STRONG_SKILLS_NORM, require_in_career=False)

    notice = signals.get("notice_period_days", 90)
    resp_rate = signals.get("recruiter_response_rate", 0.0)
    gh_score = signals.get("github_activity_score", 0)

    loc = profile.get("location", "").lower()
    will_relocate = signals.get("willing_to_relocate", False)
    is_local = any(city in loc for city in TARGET_CITIES)

    # Use pre-extracted, deduplicated evidence
    evidence = evidence_pair[0] if evidence_pair else None
    evidence_company = evidence_pair[1] if evidence_pair else None

    # ---- Rank-aware reasoning generation ----
    parts = []

    # Lead: career identity
    career_desc = f"{yoe:.1f}-year {title}"
    if any(kw in career_text for kw in ["startup", "0 to 1", "0-to-1", "founding", "early stage"]):
        career_desc += ", founding/startup background"
    parts.append(career_desc)

    if rank <= 20:
        # Top-20: lead with what makes them exceptional
        if core_found_career:
            parts.append(f"production experience with {', '.join(core_found_career[:3])}")
        elif core_found_claimed:
            parts.append(f"skilled in {', '.join(core_found_claimed[:3])}")
            
        if strong_found_career:
            parts.append(f"deep {', '.join(strong_found_career[:2])} expertise")
        elif strong_found_claimed:
            parts.append(f"strong foundation in {', '.join(strong_found_claimed[:2])}")
            
        if evidence and evidence_company:
            parts.append(f'at {evidence_company}: "{evidence}"')
        elif evidence:
            parts.append(f'"{evidence}"')

        # Availability strength
        if notice <= 30:
            parts.append(f"available in {notice} days")
        if resp_rate > 0.6:
            parts.append(f"{int(resp_rate*100)}% response rate")
        if gh_score > 60:
            parts.append(f"GitHub activity {gh_score:.0f}")
        if is_local:
            parts.append(f"in {profile.get('location', '')}")
        elif will_relocate:
            parts.append("willing to relocate")

    elif rank <= 50:
        # Mid-tier: balanced view with honest gaps
        if core_found_career:
            parts.append(f"has {', '.join(core_found_career[:2])}")
        elif core_found_claimed:
            parts.append(f"knows {', '.join(core_found_claimed[:2])}")
            
        if strong_found_claimed:
            parts.append(f"{', '.join(strong_found_claimed[:2])}")
        if evidence and evidence_company:
            parts.append(f'[{evidence_company}]: "{evidence}"')

        # Note gaps honestly
        gaps = []
        if notice > 60:
            gaps.append(f"{notice}-day notice")
        if resp_rate < 0.4:
            gaps.append(f"low response ({int(resp_rate*100)}%)")
        if not is_local and not will_relocate:
            gaps.append("location mismatch")
        if gaps:
            parts.append(f"concerns: {', '.join(gaps)}")

    else:
        # Rank 51-100: lead with what they have
        if strong_found_career:
            parts.append(f"deep {', '.join(strong_found_career[:2])} expertise")
        elif strong_found_claimed:
            parts.append(f"familiar with {', '.join(strong_found_claimed[:2])}")
        elif core_found_claimed:
            parts.append(f"knows {', '.join(core_found_claimed[:2])}")
        else:
            parts.append("general ML background")

        gaps = []
        if notice > 60:
            gaps.append(f"{notice}-day notice")
        if resp_rate < 0.3:
            gaps.append(f"{int(resp_rate*100)}% response rate")
        if not is_local and not will_relocate:
            gaps.append("location mismatch")
        if not core_found_career:
            gaps.append("lacks core retrieval scale evidence")
        if gaps:
            parts.append(f"gaps: {', '.join(gaps)}")
        if evidence and evidence_company:
            parts.append(f'but has evidence [{evidence_company}]: "{evidence}"')

    story = ", ".join(parts)
    return f"Score {score:.3f}: {story}."


In [ ]:
%%writefile main.py
import sys
import os
import time
import argparse
import csv
from data_loader import load_candidates_stream
from plausibility_filter import is_honeypot, get_penalty_reasons
from profile_builder import build_candidate_text
from scorer import score_candidate
from reasoning_generator import generate_reasoning

# Disable all network access to enforce strict offline execution
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

# ML-relevant keywords with weights for cheap pre-scoring
JD_KEYWORDS = {
    "machine learning": 5, "deep learning": 5, "nlp": 4, "natural language": 4,
    "pytorch": 5, "tensorflow": 4, "scikit": 3, "xgboost": 3,
    "embedding": 5, "vector": 4, "faiss": 5, "rag": 5,
    "retrieval": 4, "ranking": 4, "recommendation": 3,
    "neural": 4, "transformer": 5, "bert": 5, "llm": 4,
    "fine-tun": 4, "lora": 5, "peft": 5, "qlora": 5,
    "hugging": 3, "search engine": 3, "information retrieval": 4,
    "data scien": 3, "computer vision": 3,
    "pinecone": 5, "weaviate": 5, "qdrant": 5, "milvus": 5,
    "opensearch": 4, "elasticsearch": 3,
    "sentence-transformers": 5, "cosine similarity": 4,
    "python": 2, "aws": 1, "docker": 1, "mlops": 3,
    "ndcg": 5, "mrr": 5, "map": 5,
    "a/b test": 4, "evaluation framework": 4, "hybrid search": 4, "bge": 4,
}

CORE_KEYWORDS = [
    "embedding", "vector", "faiss", "rag", "retrieval", "ndcg", "mrr", 
    "transformer", "bert", "llm", "sentence-transformers", "pinecone", 
    "weaviate", "qdrant", "milvus", "opensearch",
    "nearest neighbour", "ann index", "approximate nearest", "dense retrieval",
    "semantic search", "bi-encoder", "cross-encoder", "reranker", "nmslib", "scann"
]

SHORTLIST_SIZE = 3500


def cheap_keyword_score(text_lower):
    """Fast keyword-weighted score - no AI model involved."""
    score = 0
    for keyword, weight in JD_KEYWORDS.items():
        if keyword in text_lower:
            score += weight
    return score


def main():
    parser = argparse.ArgumentParser(description="Redrob AI Candidate Ranking System")
    parser.add_argument("--candidates", type=str, default="candidates.jsonl", help="Path to candidates.jsonl")
    parser.add_argument("--out", type=str, default="team_TeamClover.csv", help="Output CSV path")
    parser.add_argument("--audit-honeypots", action="store_true", help="Audit honeypots and write to honeypot_audit.csv")
    parser.add_argument("--debug-stage1", action="store_true", help="Write stage1_debug.csv")
    args = parser.parse_args()

    start_time = time.time()

    if args.audit_honeypots:
        print("Auditing honeypots...", flush=True)
        from collections import Counter
        reason_counts = Counter()
        with open("honeypot_audit.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["candidate_id", "reason_code"])
            for candidate in load_candidates_stream(args.candidates):
                is_hp, reason = is_honeypot(candidate)
                if is_hp:
                    reason_counts[reason] += 1
                    writer.writerow([candidate.get("candidate_id"), reason])

        print("Honeypot Audit Results:", flush=True)
        for reason, count in reason_counts.most_common():
            print(f"  {reason}: {count}")
        return

    # ---- Stage 1: Cheap keyword scan on ALL 100k candidates ----
    print("Stage 1: Fast keyword scan on all candidates...", flush=True)
    shortlist = []
    total = 0
    honeypots = 0

    for candidate in load_candidates_stream(args.candidates):
        total += 1
        if total % 20000 == 0:
            print(f"  ... scanned {total}", flush=True)

        is_hp, reason = is_honeypot(candidate)
        if is_hp:
            honeypots += 1
            continue

        text_profile = build_candidate_text(candidate)
        text_lower = text_profile.lower()
        kscore = cheap_keyword_score(text_lower)

        if kscore > 0:
            has_core = any(ck in text_lower for ck in CORE_KEYWORDS)

            # Discount keyword-stuffers: if core keywords appear in skills but
            # not in career descriptions, halve the keyword score
            career_desc_text = " ".join(
                j.get("description", "").lower() for j in candidate.get("career_history", [])
            )
            core_in_career = any(ck in career_desc_text for ck in CORE_KEYWORDS)
            if has_core and not core_in_career:
                kscore = kscore // 2

            shortlist.append((kscore, candidate, text_profile[:600], has_core))

    print(f"Stage 1 done: {total} scanned, {honeypots} honeypots, {len(shortlist)} have ML keywords", flush=True)

    shortlist.sort(key=lambda x: x[0], reverse=True)

    # Two-bucket shortlist
    bucket_1_size = int(SHORTLIST_SIZE * 0.8)
    bucket_1 = shortlist[:bucket_1_size]

    bucket_2_size = SHORTLIST_SIZE - bucket_1_size
    bucket_2 = []
    bucket_2_ids = set()

    for item in shortlist[bucket_1_size:]:
        if item[3]:
            bucket_2.append(item)
            bucket_2_ids.add(item[1]["candidate_id"])
            if len(bucket_2) == bucket_2_size:
                break

    if len(bucket_2) < bucket_2_size:
        needed = bucket_2_size - len(bucket_2)
        remaining_bucket = [
            x for x in shortlist[bucket_1_size:]
            if x[1]["candidate_id"] not in bucket_2_ids
        ]
        bucket_2.extend(remaining_bucket[:needed])

    final_shortlist = bucket_1 + bucket_2

    if args.debug_stage1:
        with open("stage1_debug.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["candidate_id", "kscore", "has_core_skill_keyword", "in_shortlist"])
            final_ids = {x[1]["candidate_id"] for x in final_shortlist}
            for item in shortlist:
                writer.writerow([item[1]["candidate_id"], item[0], item[3], item[1]["candidate_id"] in final_ids])

    shortlist = final_shortlist
    print(f"Shortlisted top {len(shortlist)} for AI embedding", flush=True)

    stage1_time = time.time()
    print(f"Stage 1 took {stage1_time - start_time:.1f}s", flush=True)

    # ---- Stage 2: AI embedding on shortlisted candidates only ----
    print("Stage 2: Loading AI model...", flush=True)
    from embedding_engine import EmbeddingEngine
    engine = EmbeddingEngine()
    print("Model loaded. Encoding shortlisted candidates...", flush=True)

    texts = [item[2] for item in shortlist]
    candidates = [item[1] for item in shortlist]

    batch_size = 1024
    all_scores = []
    for batch_start in range(0, len(texts), batch_size):
        batch_end = min(batch_start + batch_size, len(texts))
        batch_texts = texts[batch_start:batch_end]
        batch_cands = candidates[batch_start:batch_end]

        embeddings = engine.batch_encode(batch_texts, batch_size=batch_size)
        semantic_scores = engine.compute_similarity(embeddings)

        for i, c in enumerate(batch_cands):
            pen = get_penalty_reasons(c)
            final_score, sem_s, sig_s, reason = score_candidate(c, semantic_scores[i], penalty_reasons=pen)
            all_scores.append({
                "candidate_id": c["candidate_id"],
                "score": final_score,
                "semantic": sem_s,
                "signal": sig_s,
                "candidate": c
            })

        print(f"  ... embedded {batch_end}/{len(texts)}", flush=True)

    # Sort descending by score, ascending by candidate_id for tiebreak (submission spec)
    all_scores.sort(key=lambda x: (-x["score"], x["candidate_id"]))
    
    # ---- Stage 3: Cross-Encoder Re-ranking on Top 300 ----
    print("Stage 3: Cross-Encoder Re-ranking...", flush=True)
    top_300 = all_scores[:300]
    local_cross_path = os.path.join(os.path.dirname(__file__), "local_model", "ms-marco-MiniLM-L-6-v2")

    if not os.path.isdir(local_cross_path):
        print(f"  WARN: Cross-Encoder model not found at {local_cross_path}.", flush=True)
        print("  Run 'python download_model.py' first. Falling back to Bi-Encoder ranking.", flush=True)
        top_100 = all_scores[:100]
    else:
        try:
            from sentence_transformers import CrossEncoder
            cross_encoder = CrossEncoder(local_cross_path, max_length=512)

            query = (
                "Senior AI Engineer Founding Team Production Embeddings Vector Database "
                "Python Evaluation Shipped Code A/B Test"
            )
            cross_inp = []
            for item in top_300:
                c = item["candidate"]
                career_text = " ".join([j.get("description", "") for j in c.get("career_history", [])])
                doc = c.get("profile", {}).get("summary", "") + " " + career_text
                cross_inp.append([query, doc[:1500]])

            print("  Running Cross-Encoder prediction...", flush=True)
            cross_scores = cross_encoder.predict(cross_inp)
            
            min_cross = float(min(cross_scores))
            max_cross = float(max(cross_scores))
            range_cross = max_cross - min_cross if max_cross > min_cross else 1.0

            for idx, item in enumerate(top_300):
                norm_cross = (cross_scores[idx] - min_cross) / range_cross
                item["score"] = item["score"] * 0.6 + norm_cross * 0.4

            top_300.sort(key=lambda x: (-x["score"], x["candidate_id"]))
            top_100 = top_300[:100]
            print("  Stage 3 complete.", flush=True)
        except Exception as e:
            print(f"  ERROR in Stage 3: {e}", flush=True)
            print("  Falling back to Bi-Encoder ranking.", flush=True)
            top_100 = all_scores[:100]

    # Generate CSV with deduplicated evidence sentences
    print(f"Writing top 100 to {args.out}...", flush=True)
    from profile_builder import extract_evidence_sentence
    seen_evidence = set()

    with open(args.out, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["candidate_id", "rank", "score", "reasoning"])

        for rank, item in enumerate(top_100, start=1):
            evidence, company = extract_evidence_sentence(item["candidate"])
            evidence_pair = None
            if evidence and evidence not in seen_evidence:
                seen_evidence.add(evidence)
                evidence_pair = (evidence, company)

            reasoning = generate_reasoning(
                item["candidate"], rank, item["score"], item["semantic"], item["signal"],
                evidence_pair=evidence_pair
            )
            writer.writerow([item["candidate_id"], rank, float(item["score"]), reasoning])

    end_time = time.time()
    print(f"Finished in {end_time - start_time:.1f}s (Stage 1: {stage1_time - start_time:.1f}s, Stage 2: {end_time - stage1_time:.1f}s)", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# 1. Download the AI models first (to satisfy offline constraint)
!python download_model.py

# 2. Run the actual pipeline completely offline!
!python main.py --candidates test_candidates.jsonl

In [ ]:
import pandas as pd
try:
    df = pd.read_csv("team_TeamClover.csv")
    # Increase column width so reasoning can be read
    pd.set_option("display.max_colwidth", None)
    display(df)
except Exception as e:
    print("Could not read output CSV. Did you upload test_candidates.jsonl?")
